🔷 What is Layer Normalization?


Layer Normalization (LayerNorm) is a technique used in Transformer models (GPT, BERT, LLaMA, etc.) to stabilize training by normalizing the activations of each token independently across its feature dimension.

In [1]:
import torch
import torch.nn as nn

# ── Built-in LayerNorm ──────────────────────────────────────────
d_model = 512
layer_norm = nn.LayerNorm(d_model)

x = torch.randn(2, 10, d_model)  # (batch=2, seq_len=10, features=512)
out = layer_norm(x)
print(out.shape)  # torch.Size([2, 10, 512])


# ── Manual Implementation (to understand internals) ─────────────
class LayerNormManual(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))   # scale
        self.beta  = nn.Parameter(torch.zeros(d_model))  # shift
        self.eps   = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len, d_model)
        mu    = x.mean(dim=-1, keepdim=True)          # mean over features
        sigma = x.var(dim=-1, keepdim=True, unbiased=False)  # variance
        x_hat = (x - mu) / torch.sqrt(sigma + self.eps)      # normalize
        return self.gamma * x_hat + self.beta                 # scale & shift


# ── Minimal Transformer Block using Pre-LN ─────────────────────
class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ln2  = nn.LayerNorm(d_model)
        self.ff   = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Pre-LN attention sub-layer
        normed, _ = self.attn(self.ln1(x), self.ln1(x), self.ln1(x))
        x = x + normed
        # Pre-LN feed-forward sub-layer
        x = x + self.ff(self.ln2(x))
        return x


# ── Quick test ──────────────────────────────────────────────────
model = TransformerBlock(d_model=512, n_heads=8, d_ff=2048)
x     = torch.randn(2, 10, 512)
print(model(x).shape)  # torch.Size([2, 10, 512])

torch.Size([2, 10, 512])
torch.Size([2, 10, 512])


In [2]:
import torch
import torch.nn as nn

# ── Built-in LayerNorm ──────────────────────────────────────────
d_model = 512
layer_norm = nn.LayerNorm(d_model)
x = torch.randn(2, 10, d_model)  # (batch=2, seq_len=10, features=512)
out = layer_norm(x)
print(out.shape)  # torch.Size([2, 10, 512])


# ── Manual Implementation (to understand internals) ─────────────
class LayerNormManual(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))   # scale
        self.beta  = nn.Parameter(torch.zeros(d_model))  # shift
        self.eps   = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len, d_model)
        mu    = x.mean(dim=-1, keepdim=True)                  # mean over features
        sigma = x.var(dim=-1, keepdim=True, unbiased=False)   # variance
        x_hat = (x - mu) / torch.sqrt(sigma + self.eps)       # normalize
        return self.gamma * x_hat + self.beta                  # scale & shift


# ── Minimal Transformer Block using Pre-LN ─────────────────────
class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ln2  = nn.LayerNorm(d_model)
        self.ff   = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Pre-LN attention sub-layer
        normed, _ = self.attn(self.ln1(x), self.ln1(x), self.ln1(x))
        x = x + normed
        # Pre-LN feed-forward sub-layer
        x = x + self.ff(self.ln2(x))
        return x


# ── Quick test ──────────────────────────────────────────────────
model = TransformerBlock(d_model=512, n_heads=8, d_ff=2048)
x     = torch.randn(2, 10, 512)
print(model(x).shape)  # torch.Size([2, 10, 512])


# ════════════════════════════════════════════════════════════════
# EXTENDED CODE — NEW SECTIONS BELOW
# ════════════════════════════════════════════════════════════════


# ── 1. Verify Manual vs Built-in LayerNorm give same result ─────
#
# Confirms our manual implementation is correct by comparing
# its output to PyTorch's official nn.LayerNorm on the same input.

def verify_manual_vs_builtin():
    d = 64
    x = torch.randn(2, 5, d)

    builtin = nn.LayerNorm(d)
    manual  = LayerNormManual(d)
    manual.gamma = builtin.weight
    manual.beta  = builtin.bias

    out_builtin = builtin(x)
    out_manual  = manual(x)

    max_diff = (out_builtin - out_manual).abs().max().item()
    print(f"[Verify] Max difference built-in vs manual: {max_diff:.2e}")
    # Should be ~1e-7 (just floating-point noise)

verify_manual_vs_builtin()


# ── 2. RMSNorm — used in LLaMA, Mistral, Gemma ─────────────────
#
# Simplified LayerNorm that removes mean subtraction.
# Faster and equally effective in large LLMs.
#
# Formula:  RMSNorm(x) = x / RMS(x) * γ
#           RMS(x) = sqrt( mean(x²) + ε )

class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.eps   = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return self.gamma * (x / rms)


# ── 3. Post-LN vs Pre-LN side-by-side ───────────────────────────
#
# Post-LN (original 2017 Transformer):  x = LayerNorm(x + Sublayer(x))
# Pre-LN  (modern GPT / LLaMA style):   x = x + Sublayer(LayerNorm(x))
#
# Pre-LN is much more stable for deep models.

class TransformerBlockPostLN(nn.Module):
    """Original 2017 paper — LayerNorm AFTER residual addition."""
    def __init__(self, d_model: int, n_heads: int, d_ff: int):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ln2  = nn.LayerNorm(d_model)
        self.ff   = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        x = self.ln1(x + attn_out)    # norm AFTER adding
        x = self.ln2(x + self.ff(x))  # norm AFTER adding
        return x


class TransformerBlockPreLN(nn.Module):
    """Modern GPT / LLaMA — LayerNorm BEFORE sublayer."""
    def __init__(self, d_model: int, n_heads: int, d_ff: int):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ln2  = nn.LayerNorm(d_model)
        self.ff   = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        attn_out, _ = self.attn(self.ln1(x), self.ln1(x), self.ln1(x))
        x = x + attn_out              # norm BEFORE sublayer
        x = x + self.ff(self.ln2(x))  # norm BEFORE sublayer
        return x


# ── 4. Stacked Transformer — multiple blocks in sequence ─────────
#
# Real models stack many transformer blocks (GPT-3 uses 96 layers).
# Each block: LayerNorm → Attention → residual
#             LayerNorm → FeedForward → residual

class StackedTransformer(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, n_layers: int):
        super().__init__()
        self.layers   = nn.ModuleList([
            TransformerBlockPreLN(d_model, n_heads, d_ff)
            for _ in range(n_layers)
        ])
        self.final_ln = nn.LayerNorm(d_model)  # final norm after all blocks

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return self.final_ln(x)


# ── 5. Inspect statistics before and after LayerNorm ────────────
#
# Shows concretely what LayerNorm does:
#   before → mean and std can be anything
#   after  → mean ≈ 0, std ≈ 1  (per token)

def inspect_stats():
    d     = 512
    x     = torch.randn(1, 4, d) * 50 + 10  # intentionally large mean/std
    ln    = nn.LayerNorm(d)
    x_out = ln(x)

    print("\n[Stats] Per-token mean and std BEFORE LayerNorm:")
    for i in range(4):
        print(f"  Token {i}: mean={x[0,i].mean().item():+.2f},  std={x[0,i].std().item():.2f}")

    print("\n[Stats] Per-token mean and std AFTER LayerNorm:")
    for i in range(4):
        print(f"  Token {i}: mean={x_out[0,i].mean().item():+.6f},  std={x_out[0,i].std().item():.6f}")
    # mean ≈ 0.000000, std ≈ 1.000000 for every token

inspect_stats()


# ── 6. Full mini language model: embedding + stacked transformer ─
#
# Puts everything together:
#   token IDs → embedding → stacked transformer blocks → output logits
# This is the skeleton of GPT-style models.

class MiniGPT(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, n_heads: int,
                 d_ff: int, n_layers: int, max_seq_len: int):
        super().__init__()
        self.token_emb   = nn.Embedding(vocab_size, d_model)
        self.pos_emb     = nn.Embedding(max_seq_len, d_model)
        self.transformer = StackedTransformer(d_model, n_heads, d_ff, n_layers)
        self.head        = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        B, T = token_ids.shape
        pos  = torch.arange(T, device=token_ids.device).unsqueeze(0)  # (1, T)
        x    = self.token_emb(token_ids) + self.pos_emb(pos)          # (B, T, d_model)
        x    = self.transformer(x)                                     # (B, T, d_model)
        return self.head(x)                                            # (B, T, vocab_size)


# ── Final end-to-end test ────────────────────────────────────────
print("\n[MiniGPT] Building model...")
mini_gpt = MiniGPT(
    vocab_size=50257,   # same vocab size as GPT-2
    d_model=128,
    n_heads=4,
    d_ff=512,
    n_layers=4,
    max_seq_len=64,
)

token_ids = torch.randint(0, 50257, (2, 16))  # batch=2, seq_len=16
logits    = mini_gpt(token_ids)
print(f"[MiniGPT] Input  shape: {token_ids.shape}")   # (2, 16)
print(f"[MiniGPT] Output shape: {logits.shape}")       # (2, 16, 50257)
print("[MiniGPT] All LayerNorm layers working correctly!")

total_params = sum(p.numel() for p in mini_gpt.parameters())
print(f"[MiniGPT] Total parameters: {total_params:,}")

torch.Size([2, 10, 512])
torch.Size([2, 10, 512])
[Verify] Max difference built-in vs manual: 4.77e-07

[Stats] Per-token mean and std BEFORE LayerNorm:
  Token 0: mean=+11.51,  std=50.65
  Token 1: mean=+11.10,  std=49.66
  Token 2: mean=+9.62,  std=49.78
  Token 3: mean=+6.76,  std=48.65

[Stats] Per-token mean and std AFTER LayerNorm:
  Token 0: mean=+0.000000,  std=1.000978
  Token 1: mean=-0.000000,  std=1.000978
  Token 2: mean=+0.000000,  std=1.000978
  Token 3: mean=+0.000000,  std=1.000978

[MiniGPT] Building model...
[MiniGPT] Input  shape: torch.Size([2, 16])
[MiniGPT] Output shape: torch.Size([2, 16, 50257])
[MiniGPT] All LayerNorm layers working correctly!
[MiniGPT] Total parameters: 13,667,328
